In [ ]:
from pathlib import Path
import runpy

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pixi.toml").exists())
globals().update(runpy.run_path(str(ROOT / "src" / "repo_paths.py")))

import re
import hashlib

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from trompy import save_figure_atomic

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 10
mpl.rcParams["savefig.transparent"] = True

In [ ]:
SUMMARY_CSV = DATA_DIR / "operant_behaviour_summary.csv"
OUT_DIR = FIGS_DIR / "panels"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PELLETS = ["Chow", "Sucrose"]
PHASES = ["NR1", "PR1", "NR2"]
PHASE_LABELS = {"NR1": "NR1", "PR1": "PR", "NR2": "NR2"}
PHASE_COLOURS = {"NR1": "#EBEBEB", "PR1": "#D26959", "NR2": "#8C8C8C"}

COLORS = [PHASE_COLOURS[phase] for phase in PHASES]

jitter_x = 0.1

df = pd.read_csv(SUMMARY_CSV).copy()

if "Week3" in df.columns:
    df["Week3"] = df["Week3"].replace({"PR": "PR1"})
if "Pellet" in df.columns:
    df["Pellet"] = df["Pellet"].replace({"Sucrose ": "Sucrose"})
    df["Pellet"] = df["Pellet"].replace({"Grain": "Chow"})

df = df[df["Pellet"].isin(PELLETS) & df["Week3"].isin(PHASES)].copy()

In [ ]:
def tweak_df(df):
    
    return (
        df
        .replace({"PR": "PR1",
                  "Sucrose ": "Sucrose",
                  "Grain": "Chow"})
        .rename(columns={"MouseID": "mouse_id",
                         "Week3": "phase",
                         "Pellet": "pellet",
                         "Schedule": "schedule",
                         "PelletCount": "total_pellets",
                         "LeftPokeCount": "active_pokes",
                         "RightPokeCount": "inactive_pokes",
                         "MaxFR": "max_fr"})
        .query("pellet in @PELLETS and phase in @PHASES")
        .fillna({"total_pellets": 0, "active_pokes": 0, "inactive_pokes": 0, "max_fr": 0})
        #.assign(total_pokes)
        
    )

df = pd.read_csv(SUMMARY_CSV).copy()
df = tweak_df(df)

In [ ]:
df

In [ ]:
def remove_mice_with_zeros(df, event):
    df = df.groupby("mouse_id").filter(lambda x: (x[event] != 0).all())
    return df

def make_behav_plot(df, event, remove_zeros=False):
    if remove_zeros:
        df = remove_mice_with_zeros(df, event)

    fig, ax = plt.subplots(figsize=(1.5, 2),
                           gridspec_kw={"left": 0.4, "right": 0.95, "top": 0.95, "bottom": 0.2})

    for idx1, pellet in enumerate(PELLETS):
        df_pellet = df.query("pellet == @pellet")
        
        for idx2, phase in enumerate(PHASES):
            x = idx1*4.5 + idx2
            y = df_pellet.query("phase == @phase")[event].values
            
            x_jittered = x + np.random.uniform(-jitter_x, jitter_x, size=len(y))
            ax.scatter(x_jittered, y, label=phase, edgecolor="k", alpha=0.5, s=10, facecolor="w", clip_on=False, zorder=10, lw=0.75, )
            
            ax.bar(x, np.mean(y), color=COLORS[idx2], alpha=1, width=0.8, edgecolor="k", lw=0.75, zorder=-100)
            print(f"{pellet} {phase}: {np.mean(y)} ± {np.std(y)}")
    
    ax.set_xticks([1, 5.5], labels=PELLETS)
    sns.despine(ax=ax, offset=5)
            
    return fig, ax
    


In [ ]:
## pellets during FR1
df_to_plot = df.query("schedule == 'FR1'")

fig, ax = make_behav_plot(df_to_plot, "total_pellets", remove_zeros=True)
ax.set_ylabel("Pellets (Fixed Ratio 1)")

save_figure_atomic(fig, "fig3_FR1_pellets", OUT_DIR)

## pellets during prog ratio
df_to_plot = df.query("schedule == 'Prog'")

fig, ax = make_behav_plot(df_to_plot, "total_pellets", remove_zeros=True)
ax.set_ylabel("Pellets (Prog. Ratio)")

save_figure_atomic(fig, "fig3_progratio_pellets", OUT_DIR)

## pokes during prog ratio
df_to_plot = df.query("schedule == 'Prog'")

fig, ax = make_behav_plot(df_to_plot, "active_pokes", remove_zeros=True)
ax.set_ylabel("Active Pokes")

save_figure_atomic(fig, "fig3_progratio_pokes", OUT_DIR)


## max FR during prog ratio
## pellets during prog ratio
## pokes during prog ratio
df_to_plot = df.query("schedule == 'Prog'")

fig, ax = make_behav_plot(df_to_plot, "max_fr", remove_zeros=True)
ax.set_ylabel("Breakpoint")

save_figure_atomic(fig, "fig3_progratio_breakpoint", OUT_DIR)

In [ ]:
## Retrieval latencies

def tweak_df(df, max_pellets=6):
    
    return (df
            .rename(columns={"MouseID": "mouse_id",
                             "DietPhase": "phase",
                             "Schedule": "schedule",
                             "PelletType": "pellet",
                             "PelletIndex": "pellet_index",
                             "RetrievalLatency_s": "latency",})
            .loc[:, ["mouse_id", "phase", "schedule", "pellet", "pellet_index", "latency"]]

    )
    
DATAFOLDER = DATA_DIR
df = pd.read_csv(DATAFOLDER / "pellet_retrieval_latency.csv")
df = tweak_df(df)



In [ ]:
# get proportion of latencies under X s
def get_prop_latencies_under_x(df, x=1):
    
    return (df
            .query("pellet_index <= 6")
            .assign(under_x=lambda d: d["latency"] < x)
            )

x = 5
get_prop_latencies_under_x(df, x=x).under_x.sum() / get_prop_latencies_under_x(df, x=x).shape[0]

In [ ]:
def get_first_n_pellet_latencies(df, n=6, schedule=None, phase=None, pellet=None):
    
    return (df
            .query("schedule == @schedule and pellet_index <= @n and phase == @phase and pellet == @pellet")
            .groupby(["mouse_id"])
            .latency
            .median()
            # .apply(lambda x: x.sort_values("pellet_index").head(n))
            #.reset_index(drop=True)
           )

# think about filtering out animals with fewer than n pellets

df_fr1 = get_first_n_pellet_latencies(df, n=6, schedule="FR1", phase="NR1", pellet="Grain")
df_progratio = get_first_n_pellet_latencies(df, n=6, schedule="ProgRatio", phase="NR1", pellet="Grain")

In [ ]:
df_fr1

In [ ]:
df_fr1_nr1_grain = get_first_n_pellet_latencies(df, n=6, schedule="FR1", phase="NR1", pellet="Grain")
df_fr1_nr1_sucrose = get_first_n_pellet_latencies(df, n=6, schedule="FR1", phase="NR1", pellet="Sucrose")
df_fr1_pr_grain = get_first_n_pellet_latencies(df, n=6, schedule="FR1", phase="PR", pellet="Grain")
df_fr1_pr_sucrose = get_first_n_pellet_latencies(df, n=6, schedule="FR1", phase="PR", pellet="Sucrose")
df_fr1_nr2_grain = get_first_n_pellet_latencies(df, n=6, schedule="FR1", phase="NR2", pellet="Grain")
df_fr1_nr2_sucrose = get_first_n_pellet_latencies(df, n=6, schedule="FR1", phase="NR2", pellet="Sucrose")

In [ ]:
from scipy.stats import ttest_ind

# NR1
print(f"NR1: {ttest_ind(df_fr1_nr1_grain, df_fr1_nr1_sucrose)}")
          
# PR
print(f"PR: {ttest_ind(df_fr1_pr_grain, df_fr1_pr_sucrose)}")

# NR2
print(f"NR2: {ttest_ind(df_fr1_nr2_grain, df_fr1_nr2_sucrose)}")

In [ ]:
df

In [ ]:
max_pellets = 6

df_to_plot = (df.query("pellet_index <= @max_pellets")
                .query("schedule == 'FR1'")
                .groupby(["mouse_id", "phase", "pellet"])
                .filter(lambda x: len(x) >= max_pellets)
                .groupby(["mouse_id", "phase", "pellet"], as_index=False)
                .agg({"latency": "median"})
)

In [ ]:
def make_latency_panel(df, phase, colors=[COLORS[0], COLORS[0]]):

        fig, ax = plt.subplots(figsize=(1.5, 2),
                                gridspec_kw={"left": 0.4, "right": 0.95, "top": 0.95, "bottom": 0.2})

        grain_y = df_to_plot.query("phase == @phase and pellet == 'Grain'").latency.values
        suc_y = df_to_plot.query("phase == @phase and pellet == 'Sucrose'").latency.values

        grain_x = np.array([1]) + np.random.uniform(-jitter_x, jitter_x, size=len(grain_y))
        suc_x = np.array([2]) + np.random.uniform(-jitter_x, jitter_x, size=len(suc_y))
        
        t, p = ttest_ind(grain_y, suc_y)
        print(f"{phase}: df{len(grain_y) + len(suc_y) - 2}, t{t:.2f}, p{p:.3f}")

        ax.bar([1,2],
                [grain_y.mean(),
                suc_y.mean()],
                color=colors,
                alpha=0.5,)

        ax.scatter(grain_x, grain_y, edgecolor="k", alpha=0.5, s=10, facecolor="w", clip_on=False, zorder=10, lw=0.75)
        ax.scatter(suc_x, suc_y, edgecolor="k", alpha=0.5, s=10, facecolor="w", clip_on=False, zorder=10, lw=0.75)

        ax.set_xticks([1,2], labels=["Chow", "Suc"])
        ax.set_ylim(0, 3)
        sns.despine(ax=ax, offset=5)
        return fig, ax

NR_colors = ["#8C8C8C", "#C2C2C2"]
PR_colors = ["#C74632", "#DE9387"]

fig, ax = make_latency_panel(df_to_plot, "NR1", colors=NR_colors)
ax.set_ylabel("Retrieval Latency (s)")
save_figure_atomic(fig, "figSx_FR1_NR1_latency", OUT_DIR)

fig, ax = make_latency_panel(df_to_plot, "PR", colors=PR_colors)
save_figure_atomic(fig, "figSx_FR1_PR_latency", OUT_DIR)

fig, ax = make_latency_panel(df_to_plot, "NR2", colors=NR_colors)
save_figure_atomic(fig, "figSx_FR1_NR2_latency", OUT_DIR)

In [ ]:
def make_cumulative_latency_panel(df, phase, schedule, colors=[COLORS[0], COLORS[2]]):

        fig, ax = plt.subplots(figsize=(1.5, 2),
                                gridspec_kw={"left": 0.4, "right": 0.95, "top": 0.95, "bottom": 0.35})

        grain_lat = df.query("phase == @phase and schedule == @schedule and pellet == 'Grain'").latency.sort_values()
        cum_prob_grain = np.arange(1, len(grain_lat)+1) / len(grain_lat)
        print(f"Percent grain within 5s: {(np.sum(grain_lat <= 5.0) / len(grain_lat)) * 100:.2f}%")
        
        suc_lat = df.query("phase == @phase and schedule == @schedule and pellet == 'Sucrose'").latency.sort_values()
        cum_prob_suc = np.arange(1, len(suc_lat)+1) / len(suc_lat)
        print(f"Percent sucrose within 5s: {(np.sum(suc_lat <= 5.0) / len(suc_lat)) * 100:.2f}%")

        ax.plot(grain_lat, cum_prob_grain, color=colors[0], label="Grain", lw=2)
        ax.plot(suc_lat, cum_prob_suc, color=colors[1], label="Sucrose", lw=2)
        ax.set_xlabel("Latency (s)")
        
        ax.set_xlim(0, 5)
        ax.set_ylim(0, 1)
        ax.set_yticks([0, 0.5, 1])
        sns.despine(ax=ax, offset=5)
        return fig, ax
    
fig, ax = make_cumulative_latency_panel(df, "NR1", "FR1", colors=NR_colors)
ax.set_ylabel("Cumulative Probability")
save_figure_atomic(fig, "figSx_FR1_NR1_cumulative_latency", OUT_DIR)

fig, ax = make_cumulative_latency_panel(df, "PR", "FR1", colors=PR_colors)
save_figure_atomic(fig, "figSx_FR1_PR_cumulative_latency", OUT_DIR)

fig, ax = make_cumulative_latency_panel(df, "NR2", "FR1", colors=NR_colors)
save_figure_atomic(fig, "figSx_FR1_NR2_cumulative_latency", OUT_DIR)

all_FR1_lats = df.query("schedule == 'FR1' and pellet_index <= 6").groupby(["mouse_id", "phase", "pellet"]).filter(lambda x: len(x) >= 6)
print(f"Percent FR1 lats within 5s: {(np.sum(all_FR1_lats['latency'] <= 5.0) / len(all_FR1_lats)) * 100:.2f}%")

In [ ]:

max_pellets = 4

df_to_plot = (df.query("pellet_index <= @max_pellets")
                .query("schedule == 'ProgRatio'")
                .groupby(["mouse_id", "phase", "pellet"])
                .filter(lambda x: len(x) >= max_pellets)
                .groupby(["mouse_id", "phase", "pellet"], as_index=False)
                .agg({"latency": "median"})
)



In [ ]:
fig, ax = make_latency_panel(df_to_plot, "NR1", colors=NR_colors)
ax.set_ylabel("Retrieval Latency (s)")
save_figure_atomic(fig, "figSx_ProgRatio_NR1_latency", OUT_DIR)

fig, ax = make_latency_panel(df_to_plot, "PR", colors=PR_colors)
save_figure_atomic(fig, "figSx_ProgRatio_PR_latency", OUT_DIR)

fig, ax = make_latency_panel(df_to_plot, "NR2", colors=NR_colors)
save_figure_atomic(fig, "figSx_ProgRatio_NR2_latency", OUT_DIR)

In [ ]:
fig, ax = make_cumulative_latency_panel(df, "NR1", "ProgRatio", colors=NR_colors)
ax.set_ylabel("Cumulative Probability")
save_figure_atomic(fig, "figSx_ProgRatio_NR1_cumulative_latency", OUT_DIR)

fig, ax = make_cumulative_latency_panel(df, "PR", "ProgRatio", colors=PR_colors)
save_figure_atomic(fig, "figSx_ProgRatio_PR_cumulative_latency", OUT_DIR)

fig, ax = make_cumulative_latency_panel(df, "NR2", "ProgRatio", colors=NR_colors)
save_figure_atomic(fig, "figSx_ProgRatio_NR2_cumulative_latency", OUT_DIR)

all_ProgRatio_lats = df.query("schedule == 'ProgRatio' and pellet_index <= 6").groupby(["mouse_id", "phase", "pellet"]).filter(lambda x: len(x) >= 6)
print(f"Percent ProgRatio lats within 5s: {(np.sum(all_ProgRatio_lats['latency'] <= 5.0) / len(all_ProgRatio_lats)) * 100:.2f}%")